In [1]:
import pandas as pd
import sys
sys.path.insert(0,'..')
%load_ext autoreload

In [2]:
%autoreload
from pytorch_pretrained_bert import BertTokenizer
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from source.processing.process import addTags
from source.modeling.model_1.data import Data, batchWrapper
from source.modeling.model_1.model import Model
from source.modeling.model_1.train import trainModel
from source.modeling.model_1.score import scoreModel
from source.utility.optimizer import AdaBound, AdaBoundW
from source.utility.schedular import CosineLR

Better speed can be achieved with apex installed from https://www.github.com/nvidia/apex.


In [3]:
def train_model(name, fold, device, model, size):
    DIR = '../../data/model/{}/fold-{}/'.format(name, fold)
    FEATS_A = ['dist_a','a_url','a_cc','a_par','a_th', 'a_loc','a_cloc']
    FEATS_B = ['dist_b','b_url','b_cc','b_par','b_th', 'b_loc','b_cloc']
    train_data = pd.read_csv('../../data/process/data.tsv', sep='\t').fillna(0.)
    valid_data = pd.read_csv('../../data/process/data.tsv', sep='\t').fillna(0.)
    train_data = train_data[train_data['fold'] != fold].reset_index()
    valid_data = valid_data[valid_data['fold'] == fold].reset_index()
    print('Dataset:',train_data.shape, valid_data.shape)
    _, train_labels = addTags(train_data, True)
    _, valid_labels = addTags(valid_data, True)
    train_text = train_data[['Text','A-offset','A','B-offset','B','Pronoun-offset','Pronoun']]
    valid_text = valid_data[['Text','A-offset','A','B-offset','B','Pronoun-offset','Pronoun']]
    train_feats = (train_data[FEATS_A], train_data[FEATS_B])
    valid_feats = (valid_data[FEATS_A], valid_data[FEATS_B])
    params = {}
    params['pretrained_model_name_or_path'] = model
    params['do_lower_case'] = True
    params['never_split'] = ("[UNK]", "[SEP]", "[PAD]", "[CLS]", "[MASK]", "[A]", "[B]", "[P]")
    tokenizer = BertTokenizer.from_pretrained(**params)
    tokenizer.vocab["[A]"] = -1
    tokenizer.vocab["[B]"] = -1
    tokenizer.vocab["[P]"] = -1
    train_data = Data(tokenizer, train_text, train_feats, train_labels)
    valid_data = Data(tokenizer, valid_text, valid_feats, valid_labels)
    params = {}
    params['collate_fn'] = batchWrapper
    params['num_workers'] = 1
    params['pin_memory'] = True
    params['drop_last'] = False
    train_loader = DataLoader(dataset=train_data, batch_size=20, shuffle=True, **params)
    valid_loader = DataLoader(dataset=valid_data, batch_size=20, shuffle=False, **params)
    model = Model(model, size).float().to(device)
    optimizer = AdaBound(model.parameters(), lr=1e-3)
    params = {}
    params['train'] = train_loader
    params['valid'] = valid_loader
    params['device'] = device
    params['model'] = model
    params['optimizer'] = optimizer
    params['scheduler'] = CosineLR(optimizer, T_max=100, T_mult=0.9, eta_min=1e-4)
    params['loss_fn'] = CrossEntropyLoss()
    params['save'] = DIR
    params['epochs'] = 5
    params['batch'] = 20
    trainModel(**params)
    return None

In [4]:
def score_model(name, fold, device, model, size):
    DIR = '../../data/model/{}/fold-{}/'.format(name,fold)
    FEATS_A = ['dist_a','a_url','a_cc','a_par','a_th', 'a_loc','a_cloc']
    FEATS_B = ['dist_b','b_url','b_cc','b_par','b_th', 'b_loc','b_cloc']
    score_data = pd.read_csv('../../data/process/score.tsv', sep='\t').fillna(0)
    print('Dataset:', score_data.shape)
    _, score_labels = addTags(score_data, True)
    score_text = score_data[['Text','A-offset','A','B-offset','B','Pronoun-offset','Pronoun']]
    score_feats = (score_data[FEATS_A], score_data[FEATS_B])
    params = {}
    params['pretrained_model_name_or_path'] = model
    params['do_lower_case'] = True
    params['never_split'] = ("[UNK]", "[SEP]", "[PAD]", "[CLS]", "[MASK]", "[A]", "[B]", "[P]")
    tokenizer = BertTokenizer.from_pretrained(**params)
    tokenizer.vocab["[A]"] = -1
    tokenizer.vocab["[B]"] = -1
    tokenizer.vocab["[P]"] = -1
    score_data = Data(tokenizer, score_text, score_feats, score_labels)
    params = {}
    params['collate_fn'] = batchWrapper
    params['num_workers'] = 1
    params['pin_memory'] = True
    params['drop_last'] = False
    score_loader = DataLoader(dataset=score_data, batch_size=50, shuffle=False, **params)
    model = Model(model, size).float().to(device)
    params = {}
    params['data'] = score_loader
    params['device'] = device
    params['model'] = model
    params['path'] = DIR
    params['batch'] = 20
    scoreModel(**params)
    return None

### BERT Large - Uncased

In [5]:
train_model('model-1',1, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (1963, 38) (491, 38)


 99%|█████████▉| 1963/1980 [02:52<00:01, 14.48it/s, train_loss=0.2803, valid_loss=0.5472]


In [6]:
train_model('model-1',2, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (1963, 38) (491, 38)


 99%|█████████▉| 1963/1980 [02:55<00:01, 12.64it/s, train_loss=0.2601, valid_loss=0.5179]


In [7]:
train_model('model-1',3, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (1963, 38) (491, 38)


 99%|█████████▉| 1963/1980 [02:53<00:01, 12.85it/s, train_loss=0.2897, valid_loss=0.4889]


In [8]:
train_model('model-1',4, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (1963, 38) (491, 38)


 99%|█████████▉| 1963/1980 [02:54<00:01, 13.63it/s, train_loss=0.2887, valid_loss=0.5497]


In [9]:
train_model('model-1',5, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (1964, 38) (490, 38)


 99%|█████████▉| 1964/1980 [02:54<00:01, 15.37it/s, train_loss=0.2800, valid_loss=0.4134]


In [5]:
score_model('model-1',1, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (2000, 37)
Model Loaded: Loss: 0.5119


2000it [02:27, 16.39it/s]            


In [6]:
score_model('model-1',2, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (2000, 37)
Model Loaded: Loss: 0.4507


2000it [02:26, 16.32it/s]            


In [7]:
score_model('model-1',3, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (2000, 37)
Model Loaded: Loss: 0.4655


2000it [02:28, 16.39it/s]            


In [8]:
score_model('model-1',4, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (2000, 37)
Model Loaded: Loss: 0.4767


2000it [02:27, 16.30it/s]            


In [9]:
score_model('model-1',5, 'cuda:0', 'bert-large-uncased', 1024)

Dataset: (2000, 37)
Model Loaded: Loss: 0.4134


2000it [02:28, 16.39it/s]            
